# Efficient Transformer

这一章展示一种更接近实际训练代码的通用 Transformer 语言模型实现.

1. **现代主干组件**: 用 `RMSNorm`、`SwiGLU` 和 Weight Tying(配合 `std=0.02` 初始化) 替代标准 `LayerNorm` + FFN.
2. **Packed Sequence Training**: 用 `DataCollatorWithFlattening` 把一个 batch 的多首诗首尾拼成一条无 `<pad>` 的长序列, 消除填充浪费.
3. **变长 FlashAttention (`varlen_attn`)**: 如果在拼接后的长序列上直接构造稠密布尔 mask, 注意力复杂度会膨胀到 $O((\sum L_i)^2)$, 绝大部分算力都在计算跨诗的屏蔽区. 因此
这里改传累加长度数组 `cu_seq_lens_q` (`[0, L1, L1+L2, ..., S]`) 和 `max_length_q`, 让 PyTorch 内置的 `varlen_attn` (`window_size=(-1, 0)` 表示因果注意力) 只在每首诗内部计算, 复杂度回到真正的 $O(\sum L_i^2)$.
4. **高效训练标配**: `bfloat16` 混合精度、`AdamW(fused=True)`.

In [1]:
import torch
from torch import nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.nn.attention.varlen import varlen_attn
from transformers import DataCollatorWithFlattening

In [2]:
torch.manual_seed(0)

class RoPE(nn.Module):
    def __init__(self, head_dim, base=10000):
        super().__init__()
        inv_freq = base ** -(torch.arange(0, head_dim, 2).float() / head_dim)
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, q, k, position_ids=None):
        if position_ids is None:
            seq_len = q.size(1)
            position_ids = torch.arange(seq_len, device=q.device)[None, :]

        angles = position_ids[..., None].to(self.inv_freq.dtype) * self.inv_freq[None, None, :]
        sin = angles.sin().repeat_interleave(2, dim=-1)[:, :, None, :].to(q.dtype)
        cos = angles.cos().repeat_interleave(2, dim=-1)[:, :, None, :].to(q.dtype)
        return q * cos + self.rotate_half(q) * sin, k * cos + self.rotate_half(k) * sin
    
    @staticmethod
    def rotate_half(x):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        return torch.stack((-x2, x1), dim=-1).flatten(-2)


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)
        self.rope = RoPE(self.head_dim)

    def forward(self, x, position_ids=None, cu_seqlens=None, max_seqlen=None):
        batch_size, seq_len, d_model = x.shape
        qkv = self.qkv(x).view(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q, k = self.rope(q, k, position_ids=position_ids)

        y = varlen_attn(
            q.reshape(-1, self.num_heads, self.head_dim).contiguous(),
            k.reshape(-1, self.num_heads, self.head_dim).contiguous(),
            v.reshape(-1, self.num_heads, self.head_dim).contiguous(),
            cu_seqlens,
            cu_seqlens,
            max_seqlen,
            max_seqlen,
            window_size=(-1, 0),
        )
        y = y.reshape(batch_size, seq_len, d_model)
        return self.out(y)


class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout):
        super().__init__()
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.w3(self.dropout(F.silu(self.w1(x)) * self.w2(x)))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, ffn_dim, dropout):
        super().__init__()
        self.norm1 = nn.RMSNorm(d_model)
        self.attn = CausalSelfAttention(d_model, num_heads)
        self.norm2 = nn.RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, ffn_dim, dropout)

    def forward(self, x, position_ids=None, cu_seqlens=None, max_seqlen=None):
        x = x + self.attn(self.norm1(x), position_ids=position_ids, cu_seqlens=cu_seqlens, max_seqlen=max_seqlen)
        x = x + self.ffn(self.norm2(x))
        return x


class EfficientTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, num_heads=8, num_layers=4, ffn_dim=768, dropout=0.1, padding_idx=None):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=padding_idx)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, num_heads, ffn_dim, dropout)
            for _ in range(num_layers)
        ])
        
        self.norm = nn.RMSNorm(d_model)
        self.out = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(self._init_weights)
        self.out.weight = self.token_embedding.weight

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Embedding) and module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()

    def forward(self, input_ids, position_ids=None, cu_seqlens=None, max_seqlen=None):
        if cu_seqlens is None:
            batch_size, max_seqlen = input_ids.shape
            cu_seqlens = torch.arange(0, (batch_size + 1) * max_seqlen, step=max_seqlen, device=input_ids.device, dtype=torch.int32)
        x = self.token_embedding(input_ids)
        for layer in self.layers:
            x = layer(x, position_ids=position_ids, cu_seqlens=cu_seqlens, max_seqlen=max_seqlen)
        return self.out(self.norm(x))

In [3]:
batch_size = 128
subset_size = 50_000
max_length = 256

dataset = load_dataset("larryvrh/Chinese-Poems")
subset = dataset["train"].select(range(min(subset_size, len(dataset["train"]))))

def format_poem(poem):
    content = poem["content"].replace("\n", "").replace("。", "。\n").strip()
    text = f"{poem['title']} {poem['author']} {poem['dynasty']}\n{content}"
    return {"text": text}

text_dataset = subset.map(format_poem, desc="Formatting poems")
splits = text_dataset.train_test_split(test_size=0.05, seed=0)
train_dataset = splits["train"]
val_dataset = splits["test"]

train_texts = train_dataset["text"]
text_lengths = [len(text) for text in train_texts]
unique_chars = sorted(set("".join(train_texts)))

print(f"训练样本数: {len(train_texts)}, 验证样本数: {len(val_dataset)}")
print(f"最大文本长度: {max(text_lengths)}")
print(f"平均文本长度: {sum(text_lengths) / len(text_lengths):.2f}")
print(f"超过 max_length={max_length} 的样本比例: {sum(length > max_length for length in text_lengths) / len(text_lengths):.2%}")

训练样本数: 47500, 验证样本数: 2500
最大文本长度: 17122
平均文本长度: 95.36
超过 max_length=256 的样本比例: 2.26%


In [4]:
class CharTokenizer:
    def __init__(self, chars):
        self.id_to_token = ["<pad>", "<eos>", "<unk>"] + list(chars)
        self.id_to_text = ["", "", ""] + list(chars)
        self.token_to_id = {token: idx for idx, token in enumerate(self.id_to_token)}
        self.pad_token_id = 0
        self.eos_token_id = 1
        self.unk_token_id = 2
        self.vocab_size = len(self.id_to_token)

    def __call__(self, texts, max_length):
        return {"input_ids": [self.encode(text, max_length) for text in texts]}

    def encode(self, text, max_length):
        ids = [self.token_to_id.get(char, self.unk_token_id) for char in text[:max_length - 1]]
        return ids + [self.eos_token_id]

    def decode(self, ids):
        return "".join(self.id_to_text[int(idx)] for idx in ids)

tokenizer = CharTokenizer(unique_chars)

print("vocab_size:", tokenizer.vocab_size)

vocab_size: 10178


In [5]:
def tokenize_fn(examples):
    return tokenizer(examples["text"], max_length)

train_tokenized = train_dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing train",
)

val_tokenized = val_dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=val_dataset.column_names,
    desc="Tokenizing val",
)

collate_fn = DataCollatorWithFlattening(
    return_position_ids=True,
    return_flash_attn_kwargs=True,
    return_tensors="pt",
)

train_dataloader = DataLoader(
    train_tokenized,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
)

val_dataloader = DataLoader(
    val_tokenized,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
)

first_batch = next(iter(train_dataloader))
print(f"train batches: {len(train_dataloader)}, val batches: {len(val_dataloader)}")

boundary = first_batch["cu_seq_lens_q"][1].item()
print("first 5 cu_seq_lens_q:", first_batch["cu_seq_lens_q"][:5].tolist())
for key in ("input_ids", "labels", "position_ids"):
    print(f"{key:13s}: {first_batch[key][0, boundary-2:boundary+3].tolist()}")

print(f"train batches: {len(train_dataloader)}, val batches: {len(val_dataloader)}")

train batches: 372, val batches: 20
first 5 cu_seq_lens_q: [0, 121, 377, 421, 627]
input_ids    : [150, 1, 7274, 8370, 8980]
labels       : [150, 1, -100, 8370, 8980]
position_ids : [119, 120, 0, 1, 2]
train batches: 372, val batches: 20


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = EfficientTransformer(
    vocab_size=tokenizer.vocab_size,
    d_model=512,
    num_heads=8,
    num_layers=6,
    ffn_dim=1208,
    dropout=0.1,
    padding_idx=tokenizer.pad_token_id,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    fused=True,
)

print(f"device: {device}")
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

device: cuda
parameters: 22.64M


In [7]:
epochs = 5

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_tokens = 0
    for step, batch in enumerate(train_dataloader, start=1):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        position_ids = batch["position_ids"].to(device, non_blocking=True)
        cu_seqlens = batch["cu_seq_lens_q"].to(device, non_blocking=True)
        max_seqlen = batch["max_length_q"]

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = model(input_ids, position_ids=position_ids, cu_seqlens=cu_seqlens, max_seqlen=max_seqlen)
            target = labels[:, 1:].reshape(-1)
            loss = F.cross_entropy(
                logits[:, :-1, :].reshape(-1, logits.size(-1)),
                target,
                ignore_index=-100,
            )

        loss.backward()
        optimizer.step()

        valid = (target != -100).sum().item()
        train_loss += loss.item() * valid
        train_tokens += valid

        if step % 50 == 0:
            print(f"Epoch {epoch+1}, Step {step}, Loss {loss.item():.4f}, PPL {torch.exp(loss.detach()).item():.2f}")
    
    model.eval()
    val_loss = 0.0
    val_tokens = 0
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
        for batch in val_dataloader:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            position_ids = batch["position_ids"].to(device, non_blocking=True)
            cu_seqlens = batch["cu_seq_lens_q"].to(device, non_blocking=True)
            max_seqlen = batch["max_length_q"]

            logits = model(
                input_ids,
                position_ids=position_ids,
                cu_seqlens=cu_seqlens,
                max_seqlen=max_seqlen,
            )
            target = labels[:, 1:].reshape(-1)
            loss = F.cross_entropy(
                logits[:, :-1, :].reshape(-1, logits.size(-1)),
                target,
                ignore_index=-100,
            )

            valid = (target != -100).sum().item()
            val_loss += loss.item() * valid
            val_tokens += valid

    print(f"Epoch {epoch+1}, Train Loss: {train_loss / train_tokens:.4f}, Val Loss: {val_loss / val_tokens:.4f}")

Epoch 1, Step 50, Loss 5.8243, PPL 338.43
Epoch 1, Step 100, Loss 5.6011, PPL 270.73
Epoch 1, Step 150, Loss 5.4292, PPL 227.98
Epoch 1, Step 200, Loss 5.3063, PPL 201.60
Epoch 1, Step 250, Loss 5.1160, PPL 166.67
Epoch 1, Step 300, Loss 5.0257, PPL 152.27
Epoch 1, Step 350, Loss 4.8429, PPL 126.83
Epoch 1, Train Loss: 5.4000, Val Loss: 4.8066
Epoch 2, Step 50, Loss 4.6695, PPL 106.65
Epoch 2, Step 100, Loss 4.4990, PPL 89.93
Epoch 2, Step 150, Loss 4.3696, PPL 79.01
Epoch 2, Step 200, Loss 4.4920, PPL 89.30
Epoch 2, Step 250, Loss 4.4487, PPL 85.51
Epoch 2, Step 300, Loss 4.3772, PPL 79.62
Epoch 2, Step 350, Loss 4.3734, PPL 79.31
Epoch 2, Train Loss: 4.5042, Val Loss: 4.3289
Epoch 3, Step 50, Loss 4.2063, PPL 67.10
Epoch 3, Step 100, Loss 4.1666, PPL 64.50
Epoch 3, Step 150, Loss 4.2030, PPL 66.89
Epoch 3, Step 200, Loss 4.2174, PPL 67.85
Epoch 3, Step 250, Loss 4.1713, PPL 64.80
Epoch 3, Step 300, Loss 4.1879, PPL 65.88
Epoch 3, Step 350, Loss 4.0020, PPL 54.71
Epoch 3, Train Loss: 

## 生成与解码


In [8]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens, temperature, top_k, device=None):
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    prompt_ids = tokenizer.encode(prompt, max_length=len(prompt) + 1)[:-1]
    input_ids = torch.tensor([prompt_ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = model(input_ids)
        next_logits = logits[:, -1, :].float() / temperature

        if top_k is not None:
            values, indices = torch.topk(next_logits, top_k)
            next_logits = torch.full_like(next_logits, float("-inf"))
            next_logits.scatter_(1, indices, values)

        probs = torch.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_token], dim=1)

        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(input_ids[0])

prompts = [
    "浣溪沙 李白 唐代\n",
    "静夜思 李白 唐代\n",
    "春望 杜甫 唐代\n",
]
for prompt in prompts:
    print(generate(model, tokenizer, prompt, max_new_tokens=max_length, temperature=0.8, top_k=20))
    print("=" * 40)

浣溪沙 李白 唐代
柳絮飞飞不禁烟。
杏花深院小屏前。
燕支花影落花前。
燕过帘栊莺语急，燕飞帘幕燕飞斜。
小庭深院落花斜。
静夜思 李白 唐代
夜静思归思，幽居念远情。
月圆金井静，月入银河明。
山色生青冢，山光隔翠屏。
不知秋梦里，唯有故园人。
春望 杜甫 唐代
日暮春风吹不开，春愁犹忆酒垆来。
一枝已折春风去，一夜一回秋月来。
